# Task 2 — Machine Learning & NLP Analysis

**Dataset:** `data/fake_news_orig.csv` — misinformation / fake-news article dataset.

| Sub-task | Technique |
|----------|----------|
| 2.1 Correlation | Spearman correlation · interactive Plotly heatmap |
| 2.2 Regression | Polynomial regression (degree=2) via sklearn Pipeline |
| 2.3 Classification | Random Forest · CountVectorizer · top-20 word importances |
| 2.4 Network Graph | pyvis interactive HTML graph |
| 2.5 Word Clouds | Bigram-only word clouds with circular mask |

In [1]:
# Install required libraries
!pip install plotly scikit-learn pandas numpy wordcloud pyvis Pillow --quiet


[notice] A new release of pip is available: 25.3 -> 26.1
[notice] To update, run: pip install --upgrade pip


In [ ]:
# ── Imports ───────────────────────────────────────────────────────────────────
import os
import warnings
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from scipy import stats
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import LinearRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.feature_extraction.text import CountVectorizer, TfidfTransformer
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    classification_report, confusion_matrix, ConfusionMatrixDisplay
)
from wordcloud import WordCloud, STOPWORDS
from IPython.display import display, IFrame, HTML
import matplotlib.pyplot as plt
import nltk
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import word_tokenize
warnings.filterwarnings('ignore')

# ── Path resolution ───────────────────────────────────────────────────────────
# __vsc_ipynb_file__ is set by the VS Code kernel to the notebook's absolute path.
# Falls back to a sibling data/ folder relative to CWD when not available.
try:
    _NB_DIR = os.path.dirname(os.path.abspath(__vsc_ipynb_file__))
except NameError:
    _NB_DIR = os.getcwd()

def find_data_dir():
    '''Locate the data/ directory relative to the notebook, with CWD fallbacks.'''
    candidates = [
        os.path.join(_NB_DIR, '..', 'data'),          # sibling data/ folder
        os.path.join(os.getcwd(), 'data'),             # project root fallback
        os.path.join(_NB_DIR, 'data'),
        'data', '../data', '../../data',               # last-resort fallbacks
    ]
    for path in candidates:
        full = os.path.normpath(os.path.abspath(path))
        if os.path.isfile(os.path.join(full, 'fake_news_orig.csv')):
            return full
    raise FileNotFoundError('Cannot locate data/ directory containing fake_news_orig.csv')

DATA_DIR = find_data_dir()
print(f'Data directory: {DATA_DIR}')

# ── Load dataset ──────────────────────────────────────────────────────────────
df = pd.read_csv(os.path.join(DATA_DIR, 'fake_news_orig.csv'), low_memory=False)
print(f'Shape: {df.shape}')
print(f'Columns: {list(df.columns)}')
display(df.head(3))

Data directory: /workspaces/2026-4-6-Data-Science-for-Cyber-Security/V2/data
Shape: (12999, 20)
Columns: ['uuid', 'ord_in_thread', 'author', 'published', 'title', 'text', 'language', 'crawled', 'site_url', 'country', 'domain_rank', 'thread_title', 'spam_score', 'main_img_url', 'replies_count', 'participants_count', 'likes', 'comments', 'shares', 'type']


,uuid,ord_in_thread,author,published,title,text,language,crawled,site_url,country,domain_rank,thread_title,spam_score,main_img_url,replies_count,participants_count,likes,comments,shares,type
0,6a175f46bcd24d39b3e962ad0f29936721db70db,0,Barracuda Brigade,2016-10-26T21:41:00.000+03:00,Muslims BUSTED: They Stole Millions In Gov’t B...,Print They should pay all the back all the mon...,english,2016-10-27T01:49:27.168+03:00,100percentfedup.com,US,25689.0,Muslims BUSTED: They Stole Millions In Gov’t B...,0.0,http://bb4sp.com/wp-content/uploads/2016/10/Fu...,0,1,0,0,0,bias
1,2bdc29d12605ef9cf3f09f9875040a7113be5d5b,0,reasoning with facts,2016-10-29T08:47:11.259+03:00,Re: Why Did Attorney General Loretta Lynch Ple...,Why Did Attorney General Loretta Lynch Plead T...,english,2016-10-29T08:47:11.259+03:00,100percentfedup.com,US,25689.0,Re: Why Did Attorney General Loretta Lynch Ple...,0.0,http://bb4sp.com/wp-content/uploads/2016/10/Fu...,0,1,0,0,0,bias
2,c70e149fdd53de5e61c29281100b9de0ed268bc3,0,Barracuda Brigade,2016-10-31T01:41:49.479+02:00,BREAKING: Weiner Cooperating With FBI On Hilla...,Red State : \nFox News Sunday reported this mo...,english,2016-10-31T01:41:49.479+02:00,100percentfedup.com,US,25689.0,BREAKING: Weiner Cooperating With FBI On Hilla...,0.0,http://bb4sp.com/wp-content/uploads/2016/10/Fu...,0,1,0,0,0,bias


## Task 2.1 — Spearman Correlation Heatmap
Spearman rank correlation is used instead of Pearson because it is
robust to non-normal distributions and monotonic (non-linear) relationships.

> **Technique reference:** Spearman rank correlation is defined in [Spearman, 1904]. Computed using `scipy.stats.spearmanr` — SciPy is documented in Virtanen et al. (2020), *Nature Methods*, 17, 261–272.

In [3]:
# ── Task 2.1: Spearman Correlation ───────────────────────────────────────────
numeric_cols = [
    'ord_in_thread', 'domain_rank', 'spam_score',
    'replies_count', 'participants_count', 'likes', 'comments', 'shares'
]
# Keep only columns that actually exist in the dataset
numeric_cols = [c for c in numeric_cols if c in df.columns]

# Replace empty strings with NaN, coerce to numeric, drop missing
num_df = df[numeric_cols].apply(pd.to_numeric, errors='coerce').dropna()

# Compute Spearman correlation matrix
spearman_corr, _ = stats.spearmanr(num_df)
corr_matrix = pd.DataFrame(spearman_corr, index=num_df.columns, columns=num_df.columns)

# Interactive Plotly heatmap
fig = go.Figure(
    go.Heatmap(
        z=corr_matrix.values,
        x=corr_matrix.columns.tolist(),
        y=corr_matrix.index.tolist(),
        colorscale='RdBu',
        zmin=-1, zmax=1,
        text=corr_matrix.round(2).values,
        texttemplate='%{text}',
        hovertemplate='%{y} vs %{x}: %{z:.3f}<extra></extra>',
        colorbar=dict(title='Spearman r')
    )
)
fig.update_layout(
    title='Task 2.1 — Spearman Correlation Matrix (Fake News Dataset)',
    template='plotly_dark',
    xaxis_tickangle=-45,
    height=550
)
fig.show()

# Identify top 3 correlated pairs (exclude self-correlations)
upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
top_pairs = (
    upper.stack()
    .abs()
    .sort_values(ascending=False)
    .head(3)
)
print('Top 3 Spearman-correlated pairs:')
print(top_pairs)

Top 3 Spearman-correlated pairs:
likes          shares                1.000000
ord_in_thread  replies_count         0.984410
replies_count  participants_count    0.534235
dtype: float64


## Task 2.2 — Polynomial Regression (degree=2)
A sklearn `Pipeline` combining `PolynomialFeatures` and `LinearRegression`
models non-linear relationships between the top correlated numerical pairs.

> **Technique reference:** Polynomial regression as a flexible extension of OLS is described in [Draper & Smith, 1998]. Implementation uses scikit-learn `Pipeline` + `PolynomialFeatures` [Pedregosa et al., 2011].

In [4]:
# ── Task 2.2: Polynomial Regression ─────────────────────────────────────────
poly_pipe = Pipeline([
    ('poly', PolynomialFeatures(degree=2, include_bias=False)),
    ('linreg', LinearRegression())
])

# Use the top 3 pairs from Task 2.1
top_pair_list = list(top_pairs.index)[:3]

fig = make_subplots(
    rows=1, cols=len(top_pair_list),
    subplot_titles=[
        f'Pair {i+1}: {p[0]} vs {p[1]}' for i, p in enumerate(top_pair_list)
    ]
)

for idx, (col_x, col_y) in enumerate(top_pair_list):
    pair_df = num_df[[col_x, col_y]].dropna()
    X = pair_df[[col_x]].values
    y = pair_df[col_y].values

    poly_pipe.fit(X, y)
    x_range = np.linspace(X.min(), X.max(), 200).reshape(-1, 1)
    y_pred = poly_pipe.predict(x_range)
    r2 = poly_pipe.score(X, y)

    col_pos = idx + 1
    # Scatter: observations
    fig.add_trace(
        go.Scatter(
            x=pair_df[col_x], y=pair_df[col_y],
            mode='markers',
            marker=dict(size=4, color='#3498db', opacity=0.5),
            name='Observations',
            showlegend=(idx == 0)
        ),
        row=1, col=col_pos
    )
    # Line: polynomial regression curve
    fig.add_trace(
        go.Scatter(
            x=x_range.flatten(), y=y_pred,
            mode='lines',
            line=dict(color='#e74c3c', width=2),
            name=f'Poly Fit (R²={r2:.3f})',
            showlegend=(idx == 0)
        ),
        row=1, col=col_pos
    )
    fig.update_xaxes(title_text=col_x, row=1, col=col_pos)
    fig.update_yaxes(title_text=col_y, row=1, col=col_pos)

fig.update_layout(
    title='Task 2.2 — Polynomial Regression (degree=2) on Top Correlated Pairs',
    template='plotly_dark',
    height=450
)
fig.show()

## Task 2.3 — Naïve Bayes Text Classifier
A `MultinomialNB` Naïve Bayes classifier is trained on articles labelled
`conspiracy`, `hate`, and `satire` (80–20 train/test split).  
`CountVectorizer` (bag-of-words) + `TfidfTransformer` converts article text to feature vectors.
After training, accuracy is tested with **five samples from each type**.

> **Technique references:**  
> - TF-IDF weighting: [Salton & Buckley, 1988]  
> - Multinomial Naïve Bayes: [McCallum & Nigam, 1998]  
> - scikit-learn `MultinomialNB`: [Pedregosa et al., 2011]

In [ ]:
# ── Task 2.3: Naïve Bayes Text Classifier ───────────────────────────────────

# Filter to conspiracy, hate, satire only as specified
TARGET_TYPES = ['conspiracy', 'hate', 'satire']
clf_df = df[df['type'].isin(TARGET_TYPES)][['text', 'type']].dropna(subset=['text', 'type'])
clf_df = clf_df[clf_df['text'].str.strip() != '']
print(f'Classes: {TARGET_TYPES}')
print(f'Samples per class:\n{clf_df["type"].value_counts()}')

# Build pipeline: CountVectorizer → TF-IDF → MultinomialNB
nb_pipeline = Pipeline([
    ('vect', CountVectorizer(max_features=10000, stop_words='english', min_df=2)),
    ('tfidf', TfidfTransformer()),
    ('clf',  MultinomialNB(alpha=0.1)),
])

X_text = clf_df['text'].values
y_text = clf_df['type'].values

X_tr, X_te, y_tr, y_te = train_test_split(
    X_text, y_text, test_size=0.2, random_state=42, stratify=y_text
)

nb_pipeline.fit(X_tr, y_tr)
y_pred_nb = nb_pipeline.predict(X_te)
nb_acc = accuracy_score(y_te, y_pred_nb)

print(f'\nNaïve Bayes overall accuracy: {nb_acc:.4f}')
print('\nClassification Report:')
print(classification_report(y_te, y_pred_nb))

# Confusion matrix (Plotly)
cm_nb = confusion_matrix(y_te, y_pred_nb, labels=TARGET_TYPES)
fig_cm_nb = go.Figure(
    go.Heatmap(
        z=cm_nb,
        x=TARGET_TYPES,
        y=TARGET_TYPES,
        colorscale='Blues',
        text=cm_nb,
        texttemplate='%{text}',
        hovertemplate='True: %{y}<br>Pred: %{x}<br>Count: %{z}<extra></extra>'
    )
)
fig_cm_nb.update_layout(
    title='Task 2.3 — Confusion Matrix: Naïve Bayes (conspiracy / hate / satire)',
    xaxis_title='Predicted Label',
    yaxis_title='True Label',
    template='plotly_dark',
    height=420
)
fig_cm_nb.show()

# ── Test with 5 samples per type ─────────────────────────────────────────────
print('\n── Testing with 5 sample articles per type ──')
for t in TARGET_TYPES:
    samples = clf_df[clf_df['type'] == t]['text'].sample(5, random_state=1).values
    preds   = nb_pipeline.predict(samples)
    correct = sum(p == t for p in preds)
    print(f'  {t:>12s}: {correct}/5 correct predictions | predictions={list(preds)}')

## Task 2.4 — Interactive Network Graph (pyvis)
An author-to-article network is built with `pyvis` using **`hate`, `satire`, and `junksci`** articles.  
- `author` and `title` are nodes  
- Titles are connected to their author  
- **Node shape** represents article `type` (triangle=hate, square=satire, diamond=junksci)  
- **Node size / colour** represents `shares` on a calculated scale (0–100, 100–500, 500–1000+)  
The graph is saved to `network.html` and displayed inline.

> **Library reference:** The `pyvis` library is used for interactive HTML network graphs [Pyvis, 2021]. Graph theory background: [Hagberg et al., 2008].

In [ ]:
# ── Task 2.4: pyvis Network Graph — hate / satire / junksci ────────────────
try:
    from pyvis.network import Network

    # Filter to the three required types
    NET_TYPES = ['hate', 'satire', 'junksci']
    net_df = df[df['type'].isin(NET_TYPES)][['author', 'title', 'type', 'shares']].dropna(subset=['author', 'title'])
    net_df['shares'] = pd.to_numeric(net_df['shares'], errors='coerce').fillna(0)

    # Limit to top 10 authors by article count for readability
    top_authors = net_df['author'].value_counts().head(10).index.tolist()
    net_sample = net_df[net_df['author'].isin(top_authors)].head(80)

    # ── Shape map: node shape represents article type (per spec) ─────────────
    # pyvis shape options: triangle, square, diamond, dot, star, ellipse
    TYPE_SHAPES = {
        'hate':    'triangle',
        'satire':  'square',
        'junksci': 'diamond',
    }
    TYPE_COLOURS = {
        'hate':    '#e74c3c',
        'satire':  '#2ecc71',
        'junksci': '#f39c12',
    }

    # ── Shares → node size scale: 0–100 → 10, 100–500 → 20, 500+ → 35 ───────
    def shares_to_size(shares):
        if shares <= 100:   return 10
        elif shares <= 500: return 20
        else:               return 35

    net = Network(
        height='650px', width='100%',
        bgcolor='#1a1a2e', font_color='white'
    )
    net.barnes_hut(gravity=-8000, central_gravity=0.3)

    # Add author nodes (gold star shape)
    for author in net_sample['author'].unique():
        short_auth = str(author)[:30]
        net.add_node(
            short_auth, label=short_auth,
            color='#f1c40f', size=28, shape='star',
            title=f'Author: {author}'
        )

    # Add article nodes and edges
    for _, row in net_sample.iterrows():
        auth        = str(row['author'])[:30]
        title_short = str(row['title'])[:45]
        art_type    = str(row.get('type', 'unknown')).lower()
        colour      = TYPE_COLOURS.get(art_type, '#95a5a6')
        shape       = TYPE_SHAPES.get(art_type, 'dot')
        node_size   = shares_to_size(int(row['shares']))
        shares_band = (
            '0–100' if row['shares'] <= 100
            else '100–500' if row['shares'] <= 500
            else '500+'
        )

        if title_short not in net.node_ids:
            net.add_node(
                title_short, label=title_short,
                color=colour, size=node_size, shape=shape,
                title=f'Type: {art_type} | Shares: {int(row["shares"])} ({shares_band})'
            )
        net.add_edge(auth, title_short, color='#7f8c8d', width=1)

    output_html = 'network.html'
    net.save_graph(output_html)
    print(f'Network graph saved: {output_html}')
    print(f'Nodes: {len(net.nodes)}  |  Edges: {len(net.edges)}')
    print('Legend — Shapes: triangle=hate, square=satire, diamond=junksci')
    print('         Sizes:  small(≤100 shares), medium(101-500), large(501+)')

    display(IFrame(src=output_html, width='100%', height=680))

except ImportError:
    print('pyvis not available. Install with: pip install pyvis')
except Exception as exc:
    print(f'Network graph skipped: {exc}')

## Task 2.5 — Word Clouds with Lemmatisation (bias & conspiracy)
Pipeline for each category (`bias`, `conspiracy`):
1. **Stopword removal** — NLTK English stopwords
2. **Tokenisation** — `nltk.word_tokenize`
3. **Lemmatisation** — `WordNetLemmatizer` (noun mode)
4. **Bigram extraction** — `CountVectorizer(ngram_range=(2,2))`
5. **Word cloud** — circular mask, rendered with matplotlib

> **Technique references:**  
> - NLTK tokenisation and stopword removal: [Bird et al., 2009]  
> - WordNet lemmatisation: [Miller, 1995]  
> - WordCloud library: [Mueller, 2012]

In [ ]:
# ── Task 2.5: Word Clouds — Stopwords + Tokenisation + Lemmatisation ────────

# Download required NLTK data (safe to run multiple times)
nltk.download('wordnet', quiet=True)
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)
nltk.download('stopwords', quiet=True)

from nltk.corpus import stopwords as nltk_stopwords

STOPWORDS_SET = set(nltk_stopwords.words('english'))
lemmatizer    = WordNetLemmatizer()

def preprocess_text(text: str) -> str:
    '''
    Full NLP preprocessing pipeline:
    1. Tokenise with NLTK word_tokenize
    2. Lowercase and remove non-alpha tokens
    3. Remove stopwords
    4. Lemmatise each token (noun mode)
    Returns a cleaned, space-joined string ready for CountVectorizer.
    '''
    tokens = word_tokenize(str(text).lower())
    tokens = [t for t in tokens if t.isalpha() and t not in STOPWORDS_SET]
    tokens = [lemmatizer.lemmatize(t, pos='n') for t in tokens]
    return ' '.join(tokens)

def get_bigram_freqs(texts, max_features=200):
    '''
    Extract bigram (2-word) frequencies from preprocessed texts.
    '''
    cv = CountVectorizer(ngram_range=(2, 2), max_features=max_features)
    cv.fit(texts)
    matrix = cv.transform(texts).toarray()
    totals = matrix.sum(axis=0)
    return dict(zip(cv.get_feature_names_out(), totals))

def make_circle_mask(size=500):
    '''Create a circular mask for WordCloud.'''
    cx, cy, r = size // 2, size // 2, size // 2
    x, y = np.ogrid[:size, :size]
    outside = (x - cx)**2 + (y - cy)**2 > r**2
    return np.where(outside, 255, 0).astype(np.uint8)

# Select BIAS and CONSPIRACY articles
bias_raw        = df[df['type'].str.lower() == 'bias']['text'].dropna().tolist()
conspiracy_raw  = df[df['type'].str.lower() == 'conspiracy']['text'].dropna().tolist()

print(f'Bias articles: {len(bias_raw)} | Conspiracy articles: {len(conspiracy_raw)}')

if len(bias_raw) < 5 or len(conspiracy_raw) < 5:
    print('Not enough samples — splitting dataset 50/50 as fallback.')
    half = len(df) // 2
    bias_raw       = df['text'].dropna().iloc[:half].tolist()
    conspiracy_raw = df['text'].dropna().iloc[half:].tolist()

# Apply full preprocessing pipeline
print('Preprocessing bias texts...')
bias_processed       = [preprocess_text(t) for t in bias_raw[:3000]]
print('Preprocessing conspiracy texts...')
conspiracy_processed = [preprocess_text(t) for t in conspiracy_raw[:3000]]

circle_mask     = make_circle_mask(500)
bias_freqs      = get_bigram_freqs(bias_processed)
conspiracy_freqs = get_bigram_freqs(conspiracy_processed)

wc_bias = WordCloud(
    mask=circle_mask,
    background_color='white',
    colormap='Blues',
    contour_color='steelblue',
    contour_width=2,
    max_words=150,
    width=500, height=500
).generate_from_frequencies(bias_freqs)

wc_conspiracy = WordCloud(
    mask=circle_mask,
    background_color='white',
    colormap='Reds',
    contour_color='firebrick',
    contour_width=2,
    max_words=150,
    width=500, height=500
).generate_from_frequencies(conspiracy_freqs)

fig, axes = plt.subplots(1, 2, figsize=(14, 6), facecolor='#1a1a2e')
fig.suptitle(
    'Task 2.5 — Word Clouds (Tokenised + Lemmatised Bigrams): Bias vs Conspiracy',
    fontsize=13, color='white'
)

axes[0].imshow(wc_bias, interpolation='bilinear')
axes[0].axis('off')
axes[0].set_title('BIAS — Top Lemmatised Bigrams', color='white', fontsize=12)
axes[0].set_facecolor('#1a1a2e')

axes[1].imshow(wc_conspiracy, interpolation='bilinear')
axes[1].axis('off')
axes[1].set_title('CONSPIRACY — Top Lemmatised Bigrams', color='white', fontsize=12)
axes[1].set_facecolor('#1a1a2e')

plt.tight_layout()
plt.savefig('bigram_wordclouds.png', dpi=150, bbox_inches='tight',
            facecolor='#1a1a2e')
plt.show()
print('Word clouds saved: bigram_wordclouds.png')

## Summary

| Sub-task | Technique | Key Output |
|----------|-----------|------------|
| 2.1 | Spearman correlation | Interactive Plotly heatmap |
| 2.2 | Polynomial regression (degree=2) | sklearn Pipeline, R² per pair |
| 2.3 | Naïve Bayes (MultinomialNB) on conspiracy/hate/satire | Confusion matrix + 5-sample test |
| 2.4 | pyvis network graph (hate/satire/junksci; shape=type; size=shares) | `network.html` |
| 2.5 | Tokenisation + Lemmatisation + Bigram word clouds | `bigram_wordclouds.png` |

> **Technique references:**  
> - NLTK tokenisation and stopword removal: [Bird et al., 2009]  
> - WordNet lemmatisation: [Miller, 1995]  
> - WordCloud library: [Mueller, 2012]

---
## References

- [Spearman, 1904] Spearman, C. (1904). The Proof and Measurement of Association between Two Things. *The American Journal of Psychology*, 15(1), 72–101. https://doi.org/10.2307/1412159
- [Freedman, 2009] Freedman, D.A. (2009). *Statistical Models: Theory and Practice* (2nd ed.). Cambridge University Press.
- [Draper & Smith, 1998] Draper, N.R. & Smith, H. (1998). *Applied Regression Analysis* (3rd ed.). Wiley-Interscience.
- [Salton & Buckley, 1988] Salton, G. & Buckley, C. (1988). Term-Weighting Approaches in Automatic Text Retrieval. *Information Processing & Management*, 24(5), 513–523. https://doi.org/10.1016/0306-4573(88)90021-0
- [McCallum & Nigam, 1998] McCallum, A. & Nigam, K. (1998). A Comparison of Event Models for Naïve Bayes Text Classification. *AAAI-98 Workshop on Learning for Text Categorization*, 41–48.
- [Pyvis, 2021] Pyvis Development Team. (2021). *Pyvis: Interactive network visualizations*. GitHub. https://github.com/WestHealth/pyvis
- [Hagberg et al., 2008] Hagberg, A.A., Schult, D.A. & Swart, P.J. (2008). Exploring Network Structure, Dynamics, and Function using NetworkX. *Proceedings of the 7th Python in Science Conference (SciPy 2008)*, 11–15.
- [Bird et al., 2009] Bird, S., Klein, E. & Loper, E. (2009). *Natural Language Processing with Python*. O'Reilly Media. https://www.nltk.org/book/
- [Miller, 1995] Miller, G.A. (1995). WordNet: A Lexical Database for English. *Communications of the ACM*, 38(11), 39–41. https://doi.org/10.1145/219717.219748
- [Mueller, 2012] Mueller, A. (2012). word_cloud: A little word cloud generator in Python. GitHub. https://github.com/amueller/word_cloud
- [McKinney, 2010] McKinney, W. (2010). Data Structures for Statistical Computing in Python. *Proceedings of the 9th Python in Science Conference (SciPy)*, 51–56.
- [Harris et al., 2020] Harris, C.R., Millman, K.J., van der Walt, S.J. et al. (2020). Array programming with NumPy. *Nature*, 585, 357–362. https://doi.org/10.1038/s41586-020-2649-2
- [Pedregosa et al., 2011] Pedregosa, F., Varoquaux, G., Gramfort, A. et al. (2011). Scikit-learn: Machine Learning in Python. *Journal of Machine Learning Research*, 12, 2825–2830.
- [Plotly, 2015] Plotly Technologies Inc. (2015). *Collaborative data science*. Montréal, QC: Plotly Technologies Inc. https://plot.ly
- [Hunter, 2007] Hunter, J.D. (2007). Matplotlib: A 2D Graphics Environment. *Computing in Science & Engineering*, 9(3), 90–95. https://doi.org/10.1109/MCSE.2007.55